In [176]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


# 1. Load Data
data = pd.read_csv("/home/reda/AQI/data/raw/PRSA_Aotizhongxin.csv")

# Configuration
numeric_cols = ['PM10', 'SO2', 'NO2', 'CO', 'PM2.5', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
categorical_cols = ['wd']

# 2. Train-Test Split (First step to prevent Data Leakage)
X = data.drop('O3', axis=1)
y = data['O3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Handle Missing Values (Train Set)
X_train[numeric_cols] = X_train[numeric_cols].interpolate(limit=6)
y_train = y_train.interpolate(limit=6)

for col in categorical_cols:
    X_train[col] = X_train[col].fillna(X_train[col].mode()[0])

In [177]:
# 4. Handle Missing Values (Test Set)
X_test[numeric_cols] = X_test[numeric_cols].interpolate(limit=6)
y_test = y_test.interpolate(limit=6)

for col in categorical_cols:
    X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

In [178]:
# 5. Drop remaining NaNs (Crucial for LinearRegression)
train_mask = X_train[numeric_cols].notna().all(axis=1) & y_train.notna()
X_train = X_train[train_mask]
y_train = y_train[train_mask]

test_mask = X_test[numeric_cols].notna().all(axis=1) & y_test.notna()
X_test = X_test[test_mask]
y_test = y_test[test_mask]

In [179]:
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

In [180]:
# 6. Log Transformation (For skewed pollutants only)
skewed_cols = ['PM10', 'SO2', 'NO2', 'CO', 'PM2.5', 'RAIN', 'WSPM']
X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

In [181]:
# 7. One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [182]:
# 8. Scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [183]:
# 9. Model Fitting and Prediction
model = LinearRegression()
model.fit(X_train_scaled, y_train)
predict = model.predict(X_test_scaled)

# 10. Evaluation
r2 = r2_score(y_test, predict)
print(f"R2 Score: {r2*100:.2f}%")

R2 Score: 61.52%


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Load Data
data = pd.read_csv("/home/reda/AQI/data/raw/PRSA_Aotizhongxin.csv")

# Configuration
numeric_cols = ['PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
categorical_cols = ['wd']

# 2. Split
X = data.drop('PM2.5', axis=1)
y = data['PM2.5']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Preprocessing (Interpolation)
X_train[numeric_cols] = X_train[numeric_cols].interpolate(limit=6)
X_test[numeric_cols] = X_test[numeric_cols].interpolate(limit=6)
y_train = y_train.interpolate(limit=6)
y_test = y_test.interpolate(limit=6)

for col in categorical_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# 4. CRITICAL FIX: Sync X and y after dropping NaNs
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

# 5. Transformations
skewed_cols = ['PM10', 'SO2', 'NO2', 'CO', 'O3', 'RAIN', 'WSPM']
X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 6. Scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# 8. Prediction & Evaluation
predict = rf_model.predict(X_test_scaled)

# Now they have the same length!
r2 = r2_score(y_test, predict)
mae = mean_absolute_error(y_test, predict)

print(f"Random Forest R2 Score: {r2*100:.2f}%")
print(f"Mean Absolute Error: {mae:.2f}")

Random Forest R2 Score: 91.35%
Mean Absolute Error: 12.00


In [185]:
# 1. Load the new station data for testing
test_data_path = "/home/reda/AQI/data/raw/PRSA_Shunyi.csv" 
new_test_data = pd.read_csv(test_data_path)

# 2. Separate Features and Target
X_new = new_test_data.drop('PM2.5', axis=1)
y_new = new_test_data['PM2.5']

# 3. Preprocessing (Interpolation)
X_new[numeric_cols] = X_new[numeric_cols].interpolate(limit=6)
y_new = y_new.interpolate(limit=6)

for col in categorical_cols:
    # التعديل هنا: استخدمنا X_new بدل X_train عشان X_train العمود اتمسح منها
    X_new[col] = X_new[col].fillna(X_new[col].mode()[0]) 

# 4. FIXED: CRITICAL SYNC - Drop NaNs from both X and y
temp_df = pd.concat([X_new, y_new], axis=1).dropna()
X_new = temp_df.drop('PM2.5', axis=1)
y_new = temp_df['PM2.5']

# 5. Transformations
X_new[skewed_cols] = np.log1p(X_new[skewed_cols])
X_new = pd.get_dummies(X_new, drop_first=True)

# Ensure same columns as training
_, X_new_aligned = X_train.align(X_new, join='left', axis=1, fill_value=0)

# 6. Scaling (Use existing scaler)
X_new_scaled = scaler.transform(X_new_aligned)

# 7. Prediction
new_predict = rf_model.predict(X_new_scaled)

# 8. Evaluation
new_r2 = r2_score(y_new, new_predict)
new_mae = mean_absolute_error(y_new, new_predict)

print(f"Test Station: {test_data_path.split('/')[-1]}")
print(f"New R2 Score: {new_r2*100:.2f}%")
print(f"New MAE: {new_mae:.2f}")

Test Station: PRSA_Shunyi.csv
New R2 Score: 91.94%
New MAE: 14.40
